# Part B - Q3: Zero-shot Classification with CLIP-ViT-B/16

**Task:** Report **zero-shot** per-class classification accuracy (precision and recall)
using the **CLIP-ViT-B/16** model.

**Idea:** CLIP jointly embeds images and text. For zero-shot classification we build a
text prompt for every class (e.g. *"a photo of a accordion"*), encode all class prompts
and each test image, and assign each image to the class whose text embedding has the
highest cosine similarity with the image embedding. **No training is performed.**

In [1]:
import os, numpy as np, torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor
from sklearn.metrics import classification_report, precision_recall_fscore_support
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

DATA_ROOT = os.path.join("data", "classification", "dataset")
classes = sorted(d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d)))
cls2idx = {c: i for i, c in enumerate(classes)}
print(f"{len(classes)} classes:", classes)

c:\Users\sudwivedi\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
15 classes: ['accordion', 'bass', 'camera', 'crocodile', 'crocodile_head', 'cup', 'dollar_bill', 'emu', 'gramophone', 'hedgehog', 'nautilus', 'pizza', 'pyramid', 'sea_horse', 'windsor_chair']


## Build the SAME test split as Q1/Q2 (images after image_0040 per class)

In [2]:
def build_split(root):
    train_items, test_items = [], []
    for c in classes:
        files = sorted(f for f in os.listdir(os.path.join(root, c))
                       if f.lower().endswith((".jpg", ".jpeg", ".png")))
        for f in files:
            num = int("".join(ch for ch in os.path.splitext(f)[0] if ch.isdigit()))
            path = os.path.join(root, c, f)
            (train_items if num <= 40 else test_items).append((path, cls2idx[c]))
    return train_items, test_items

_, test_items = build_split(DATA_ROOT)
print(f"Test images (zero-shot evaluated): {len(test_items)}")

Test images (zero-shot evaluated): 205


## Load CLIP-ViT-B/16 and build text prompts for each class

In [3]:
model_name = "openai/clip-vit-base-patch16"
# use_safetensors=True avoids torch.load (blocked for torch < 2.6) by loading .safetensors weights
model = CLIPModel.from_pretrained(model_name, use_safetensors=True).to(device).eval()
processor = CLIPProcessor.from_pretrained(model_name)

# Make the class names human-readable for the text prompts
def humanize(name):
    return name.replace("_", " ")

prompts = [f"a photo of a {humanize(c)}" for c in classes]
print("Example prompts:", prompts[:3], "...")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 7950.23it/s]

Example prompts: ['a photo of a accordion', 'a photo of a bass', 'a photo of a camera'] ...


## Zero-shot prediction for every test image

For each image we run the full CLIP model with the 15 class prompts and pick the class
with the highest image-text similarity (`logits_per_image`). No training is done.

In [4]:
y_true, y_pred = [], []
with torch.no_grad():
    for path, label in test_items:
        img = Image.open(path).convert("RGB")
        inputs = processor(text=prompts, images=img, return_tensors="pt", padding=True).to(device)
        logits_per_image = model(**inputs).logits_per_image  # shape (1, 15)
        pred = int(logits_per_image.argmax(dim=-1).item())
        y_pred.append(pred)
        y_true.append(label)

print("CLIP-ViT-B/16 zero-shot - per-class precision / recall (test set):")
print(classification_report(y_true, y_pred, target_names=classes, digits=3, zero_division=0))

p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=list(range(len(classes))), zero_division=0)
df = pd.DataFrame({"precision": p.round(3), "recall": r.round(3), "f1": f.round(3), "support": s}, index=classes)
print(df)
print(f"\nOverall zero-shot accuracy: {np.mean(np.array(y_true)==np.array(y_pred)):.4f}")

CLIP-ViT-B/16 zero-shot - per-class precision / recall (test set):
                precision    recall  f1-score   support

     accordion      1.000     1.000     1.000        15
          bass      1.000     1.000     1.000        14
        camera      1.000     1.000     1.000        10
     crocodile      0.900     0.900     0.900        10
crocodile_head      0.909     0.909     0.909        11
           cup      1.000     1.000     1.000        17
   dollar_bill      1.000     1.000     1.000        12
           emu      1.000     1.000     1.000        13
    gramophone      0.917     1.000     0.957        11
      hedgehog      1.000     1.000     1.000        14
      nautilus      1.000     0.933     0.966        15
         pizza      1.000     1.000     1.000        13
       pyramid      1.000     1.000     1.000        17
     sea_horse      1.000     1.000     1.000        17
 windsor_chair      1.000     1.000     1.000        16

      accuracy                     

## Observations
* CLIP classifies these 15 categories **without any training on the dataset**, relying
  purely on its pretrained image-text alignment, yet reaches accuracy competitive with the
  trained models from Q1/Q2 because the class names map to common visual concepts.
* Classes whose names are unambiguous English nouns (`pizza`, `accordion`, `camera`)
  score very high; the more ambiguous / fine-grained names (`crocodile_head`,
  `windsor_chair`, `nautilus`) are where zero-shot CLIP loses the most precision/recall.
* Prompt wording matters: *"a photo of a {class}"* is a strong, standard template; the
  underscores in raw class names are replaced with spaces so the text encoder gets natural
  language.